In [8]:
import numpy as np
from scipy.special import softmax
import torch


In [9]:
# a 4-word sentence, hand-made embeddings (built right here)
words = ["money", "cash", "river", "bank"]
X = np.array([[3,3,0,0], [3,2.6,0,.4],   # money, cash
              [0,0,3,3], [.8,.8,2.4,2.4]])  # river, bank
d = X.shape[1]

scores  = X @ X.T / np.sqrt(d)     # similarity of every word to every word
weights = softmax(scores)          # each row -> weights that sum to 1
new_meaning = weights @ X   

In [4]:
new_meaning

array([[1.20305120e+00, 1.14622536e+00, 9.40980066e-04, 5.77668176e-02],
       [7.01066056e-01, 6.64468147e-01, 1.33849641e-03, 3.79364054e-02],
       [3.45016903e-02, 3.44784067e-02, 8.79269382e-01, 8.79292665e-01],
       [1.78923478e-02, 1.76983695e-02, 1.74510345e-01, 1.74704323e-01]])

In [5]:
scores

array([[9.  , 8.4 , 0.  , 2.4 ],
       [8.4 , 7.96, 0.6 , 2.72],
       [0.  , 0.6 , 9.  , 7.2 ],
       [2.4 , 2.72, 7.2 , 6.4 ]])

In [89]:
reviews = reviews = [
    "the movie was great",
    "i hated this movie",
    "the story was amazing",
    "the acting was excellent",
    "this movie was fantastic",
    "i really hated this",
    "the movie felt boring",
    "the story was interesting",
    "i hated the ending",
    "characters were very good"
]

# give every unique word an integer id
words = sorted({w for r in reviews for w in r.split()})
stoi = {w: i for i, w in enumerate(words)}   # string -> id

def encode(text):
    return torch.tensor([stoi[w] for w in text.split()])

print(encode("the movie was great"))   # tensor([...ids...])

tensor([16, 13, 19,  9])


In [90]:
stoi

{'acting': 0,
 'amazing': 1,
 'boring': 2,
 'characters': 3,
 'ending': 4,
 'excellent': 5,
 'fantastic': 6,
 'felt': 7,
 'good': 8,
 'great': 9,
 'hated': 10,
 'i': 11,
 'interesting': 12,
 'movie': 13,
 'really': 14,
 'story': 15,
 'the': 16,
 'this': 17,
 'very': 18,
 'was': 19,
 'were': 20}

In [91]:
import torch.nn as nn
# one trainable vector per word (here: 4 numbers each)
embedding = nn.Embedding(num_embeddings=len(stoi), embedding_dim=8)

ids = encode("the movie was great")   # shape (4,)  -> 4 words
vectors = embedding(ids)              # shape (4, 4) -> a vector per word

print(vectors.shape)   

torch.Size([4, 8])


In [52]:
vectors

tensor([[-0.1089,  1.4809,  0.5727, -1.4905,  0.5907,  2.4167,  1.4761,  1.7481],
        [-1.5490,  0.2437,  0.4614,  2.5960, -2.0873, -0.4789, -0.8364,  0.5673],
        [ 0.0676, -0.9966,  1.7634, -1.9353, -0.0257,  0.1380, -0.5452,  0.3906],
        [ 0.9579,  0.2743,  2.1377,  1.1798, -0.1633, -1.0659,  2.0444, -0.3652]],
       grad_fn=<EmbeddingBackward0>)

In [39]:
sentence_vector = vectors.mean(dim=0)   # average the rows
print(sentence_vector)          # torch.Size([4])

tensor([-0.2318,  0.9347,  1.2524, -0.6595], grad_fn=<MeanBackward1>)


In [86]:
train = [
    ("the movie was great", 1),
    ("i hated this movie", 0),
    ("the story was amazing", 1),
    ("the acting was excellent", 1),
    ("this movie was fantastic", 1),
    ("i really hated this", 0),
    ("the movie felt boring", 0),
    ("the story was interesting", 1),
    ("i hated the ending", 0),
    ("characters were very good", 1)
]

In [ ]:
import torch
import torch.nn as nn

reviews = [
    "the movie was great",
    "i hated this movie",
    "the story was amazing",
    "the acting was excellent",
    "this movie was fantastic",
    "i really enjoyed this",
    "the movie felt boring",
    "the story was interesting",
    "i hated this movie",
    "characters were very good"
]

# Get every unique word
words = sorted({w for r in reviews for w in r.split()})

# word -> integer ID
stoi = {w: i for i, w in enumerate(words)}

# Convert words to integer IDs
def encode(text):
    return torch.tensor([stoi[w] for w in text.split()])

print("Vocabulary:", words)
print("Vocabulary size:", len(stoi))

# Create embedding layer
embedding = nn.Embedding(
    num_embeddings=len(stoi),
    embedding_dim=4
)

# Encode a bigger review
review = "the movie was great"

ids = encode(review)

print("\nWord IDs:")
print(ids)

# Convert IDs into embeddings
vectors = embedding(ids)

print("\nEmbedding vectors:")
print(vectors)

# Average all word vectors
sentence_vector = vectors.mean(dim=0)

print("\nSentence vector:")
print(sentence_vector)

print("\nSentence vector shape:")
print(sentence_vector.shape)
class SentimentNet(nn.Module):
    def __init__(self, vocab_size  , dim = 0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size , dim)
        self.fc = nn.Linear(dim , 2)

    def forward(self , ids):
        pooled = self.emb(ids).mean(dim = 0 , keepdim = True)
        return self.fc(pooled)
    
        
        
train = [
    ("the movie was great", 1),
    ("i hated this movie", 0),
    ("the story was amazing", 1),
    ("the acting was excellent", 1),
    ("this movie was fantastic", 1),
    ("i really hated this", 0),
]
model = SentimentNet(len(stoi) , dim = 0)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters() , lr = 0.05)

for epoch in range(200):
    losses = [loss_fn(model(encode(t)) , torch.tensor([y])) for t , y in train]
    loss = torch.stack(losses).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"epoch : {epoch} , loss : {loss}")

prediction = model(encode("i really hated this"))
prediction
prediction = prediction.argmax().item()
prediction

In [101]:
import torch
import torch.nn as nn

class SentimentNet(nn.Module):
    # We hand the factory the dictionary when we build it
    def __init__(self, stoi, dim=8):
        super().__init__()
        self.stoi = stoi                           # The factory saves the dictionary
        self.emb = nn.Embedding(len(stoi), dim)    # Machine 1
        self.fc  = nn.Linear(dim, 2)               # Machine 2

    def forward(self, ids):
        pooled = self.emb(ids).mean(dim=0, keepdim=True)  # sentence vector
        return self.fc(pooled)   

    def fit(self, train_data, epochs=120):
        loss_fn = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=0.05)

        for epoch in range(epochs):
            losses = []
            
            # Loop through the data you passed in
            for text, true_answer in train_data: 
                
                # 1. Translate and put in a PyTorch Tensor box
                word_ids = [self.stoi[word] for word in text.split()]
                input_tensor = torch.tensor(word_ids)
                
                # 2. Make a guess using self() 
                guess = self(input_tensor) 
                
                # 3. Grade the guess
                truth = torch.tensor([true_answer])
                error = loss_fn(guess, truth)
                losses.append(error)

            # 4. The 3 Magical Steps
            loss = torch.stack(losses).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        print("Training Complete!")

    def predict(self, sentence: str):
        # 1. Translate the sentence and put it in a Tensor box
        word_ids = [self.stoi[word] for word in sentence.split()]
        input_tensor = torch.tensor(word_ids)
        
        # 2. Turn off the "Blame Game" (gradients) to save memory
        with torch.no_grad():
            raw_scores = self(input_tensor)
            
        # 3. Ask the Judge which score is higher
        prediction_id = raw_scores.argmax().item()
        
        return "Positive" if prediction_id == 1 else "Negative"

In [94]:
# model     = SentimentNet(stoi, dim=8)
# loss_fn   = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

# for epoch in range(120):
#     losses = [loss_fn(model(encode(t)), torch.tensor([y]))
#               for t, y in train]        # 1. forward every review
#     loss = torch.stack(losses).mean()  # 2. how wrong
#     optimizer.zero_grad()              # 3.
#     loss.backward()                    # 4.
#     optimizer.step()                   # 5. update
#     print(f"epoch: {epoch} loss: {loss}")

epoch: 0 loss: 0.6795094609260559
epoch: 1 loss: 0.5842862725257874
epoch: 2 loss: 0.5048487782478333
epoch: 3 loss: 0.436014324426651
epoch: 4 loss: 0.37191253900527954
epoch: 5 loss: 0.3093581795692444
epoch: 6 loss: 0.24958594143390656
epoch: 7 loss: 0.19555625319480896
epoch: 8 loss: 0.14940504729747772
epoch: 9 loss: 0.11150012165307999
epoch: 10 loss: 0.0812949538230896
epoch: 11 loss: 0.057988982647657394
epoch: 12 loss: 0.04059721156954765
epoch: 13 loss: 0.02802705205976963
epoch: 14 loss: 0.019197262823581696
epoch: 15 loss: 0.013136237859725952
epoch: 16 loss: 0.009041075594723225
epoch: 17 loss: 0.006295790430158377
epoch: 18 loss: 0.004456097260117531
epoch: 19 loss: 0.0032158945687115192
epoch: 20 loss: 0.002370771486312151
epoch: 21 loss: 0.0017867106944322586
epoch: 22 loss: 0.0013763975584879518
epoch: 23 loss: 0.001083088107407093
epoch: 24 loss: 0.0008696977747604251
epoch: 25 loss: 0.0007116454071365297
epoch: 26 loss: 0.0005925713339820504
epoch: 27 loss: 0.0005013

In [102]:
my_model = SentimentNet(stoi, dim=8)

my_model.fit(train)

Training Complete!


In [103]:
prediction = my_model.predict("the movie was great")

In [104]:
prediction

'Positive'